In [ ]:
from __future__ import annotations
from pathlib import Path
from itertools import product
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
import joblib
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", category=UserWarning)


# -----------------------------
# IO + Alignment utilities
# -----------------------------

def _assert_aligned_obs(*adatas: ad.AnnData):
    base = adatas[0].obs_names.astype(str).to_numpy()
    for k, A in enumerate(adatas[1:], start=1):
        cur = A.obs_names.astype(str).to_numpy()
        if not np.array_equal(base, cur):
            raise ValueError(f"obs_names mismatch between modality 0 and modality {k}. "
                             f"Example: {base[:3]} vs {cur[:3]}")

def load_regression_data(target_name: str, base_dir: str = "../data") -> dict:
    """Loads RNA, ATAC, ADT-minus; loads y from response CSV and aligns by cell IDs."""
    data_path = Path(base_dir)
    resp_path = data_path / "response"

    rna = ad.read_h5ad(data_path / "rna.h5ad")
    atac = ad.read_h5ad(data_path / "atac.h5ad")
    adt_minus = ad.read_h5ad(data_path / f"adt_minus_{target_name}.h5ad")

    _assert_aligned_obs(rna, atac, adt_minus)

    cell_ids = rna.obs_names.astype(str)
    y_df = pd.read_csv(resp_path / f"{target_name}.csv", index_col=0)
    y_vec = y_df.loc[cell_ids].iloc[:, 0].to_numpy(dtype=float)

    return {"rna": rna, "atac": atac, "adt_minus": adt_minus, "y": y_vec, "cell_ids": cell_ids}

def load_train_indices(splits_dir: str, split_tag: str, n_cells: int) -> np.ndarray:
    path = Path(splits_dir) / f"{split_tag}_train_idx.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing: {path}")

    df = pd.read_csv(path)
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not num_cols:
        raise ValueError(f"No numeric column found in {path}")

    idx = df[num_cols[0]].to_numpy()

    if np.isnan(idx).any():
        raise ValueError(f"NaNs found in {path}")
    idx_int = idx.astype(np.int64)
    if not np.allclose(idx, idx_int):
        raise ValueError(f"Non-integer indices found in {path}")
    idx = idx_int

    mn, mx = int(idx.min()), int(idx.max())

    if mn == 1 and mx == n_cells:
        idx0 = idx - 1
    else:
        idx0 = idx

    if (idx0 < 0).any() or (idx0 >= n_cells).any():
        raise ValueError(
            f"Out-of-range indices in {path}: "
            f"min={idx0.min()}, max={idx0.max()}, n_cells={n_cells}"
        )

    return idx0

def get_norm(adata: ad.AnnData):
    X = adata.layers["norm"] if "norm" in adata.layers else adata.X
    return X.tocsr() if sp.issparse(X) else np.asarray(X)


# -----------------------------
# Center+scale preprocessing (no dimensionality reduction)
# -----------------------------

def fit_transform_view(X_tr, X_val, seed: int = 0):
    """Convert sparse to dense, then center+scale. No SVD/PCA reduction."""
    X_tr2 = X_tr.toarray() if sp.issparse(X_tr) else np.asarray(X_tr)
    X_va2 = X_val.toarray() if sp.issparse(X_val) else np.asarray(X_val)

    scaler = StandardScaler(with_mean=True, with_std=True)
    X_tr2 = scaler.fit_transform(X_tr2)
    X_va2 = scaler.transform(X_va2)

    return X_tr2, X_va2, {"scaler": scaler, "type": "scaler"}


def soft_threshold(A: np.ndarray, tau: float) -> np.ndarray:
    return np.sign(A) * np.maximum(np.abs(A) - tau, 0.0)


# -----------------------------
# Cooperative regression
# -----------------------------

def coop_regression_fit_scaled(
    X_views_tr, y_tr: np.ndarray,
    rho: float, lambdas: list[float],
    X_views_val=None, y_val: np.ndarray | None = None,
    step_size: float = 1e-2, max_iter: int = 500,
    fista: bool = True, random_state: int = 0,
    patience: int = 25, min_delta: float = 1e-6,
):
    rng = np.random.default_rng(random_state)
    n = y_tr.shape[0]
    M = len(X_views_tr)
    p_dims = [X.shape[1] for X in X_views_tr]

    y_tr = y_tr.reshape(-1, 1)
    if y_val is not None:
        y_val = y_val.reshape(-1, 1)

    Thetas = [0.01 * rng.standard_normal((p_dims[m], 1)) for m in range(M)]
    Zs = [T.copy() for T in Thetas]
    t_k = 1.0
    best_metric, best_Thetas, wait = np.inf, None, 0

    for it in range(1, max_iter + 1):
        cur = Zs if fista else Thetas
        Fs = [X_views_tr[m] @ cur[m] for m in range(M)]
        Fsum = sum(Fs)
        residual = Fsum - y_tr

        new_Thetas = []
        for m in range(M):
            grad = (X_views_tr[m].T @ residual) / n
            grad += (rho / n) * (X_views_tr[m].T @ (M * Fs[m] - Fsum))
            new_Thetas.append(
                soft_threshold(cur[m] - step_size * grad, step_size * (lambdas[m] / p_dims[m]))
            )

        if fista:
            t_new = 0.5 * (1.0 + np.sqrt(1.0 + 4.0 * t_k**2))
            Zs = [
                new_Thetas[m] + ((t_k - 1.0) / t_new) * (new_Thetas[m] - Thetas[m])
                for m in range(M)
            ]
            t_k = t_new

        Thetas = new_Thetas

        if X_views_val is not None:
            y_hat_val = sum(X_views_val[m] @ Thetas[m] for m in range(M))
            val_mse = mean_squared_error(y_val, y_hat_val)
            if val_mse < best_metric - min_delta:
                best_metric, best_Thetas, wait = val_mse, [T.copy() for T in Thetas], 0
            else:
                wait += 1
            if wait >= patience and best_Thetas is not None:
                Thetas = best_Thetas
                break

    return Thetas


# -----------------------------
# Parallel grid search
# -----------------------------

def _evaluate_one(X_tr, y_tr, X_val, y_val, rho, lams, seed):
    """Evaluate a single (rho, lambdas) combination. Top-level for joblib pickling."""
    Thetas = coop_regression_fit_scaled(
        X_tr, y_tr, rho, lams, X_views_val=X_val, y_val=y_val,
        random_state=seed
    )
    y_pred = sum(X_val[m] @ Thetas[m] for m in range(len(X_tr)))
    return rho, lams, float(mean_squared_error(y_val, y_pred))


def holdout_select_params_reg(X_tr, y_tr, X_val, y_val, rho_grid, lam_grids, seed=0, n_jobs=4):
    combos = list(product(rho_grid, lam_grids[0], lam_grids[1], lam_grids[2]))
    total = len(combos)
    print(f"Starting Grid Search: {total} combinations on {n_jobs} cores")

    best, best_mse, cnt = None, np.inf, 0
    for rho, lams, mse in Parallel(n_jobs=n_jobs, verbose=0, prefer="threads",
                                    return_as="generator_unordered")(
        delayed(_evaluate_one)(X_tr, y_tr, X_val, y_val, rho, [l0, l1, l2], seed)
        for rho, l0, l1, l2 in combos
    ):
        cnt += 1
        print(f"[{cnt}/{total}] rho={rho}, lams={lams} | Val MSE={mse:.6f}")
        if mse < best_mse:
            best_mse = mse
            best = {"rho": rho, "lambdas": lams, "val_mse": mse}
            print("  --> New best")

    return best


# -----------------------------
# Main pipeline
# -----------------------------

def train_regression_pipeline(
    target_name: str,
    dataset_name: str = "tea",
    split_tag: str = "tea_split3_all_celltypes",
    base_dir: str = "../data",
    splits_dir: str = "../splits",
    val_ratio: float = 0.2,
    seed: int = 0,
    n_jobs: int = 4,
):
    model_root = Path(f"../models/{dataset_name}/{split_tag}_{target_name}_coopreg")
    model_root.mkdir(parents=True, exist_ok=True)

    bundle = load_regression_data(target_name, base_dir=base_dir)
    y_all = bundle["y"]

    n_cells = bundle["rna"].n_obs
    train_idx_global = load_train_indices(splits_dir, split_tag, n_cells)

    tr_idx, val_idx = train_test_split(train_idx_global, test_size=val_ratio, random_state=seed)

    X_rna  = get_norm(bundle["rna"])
    X_adtm = get_norm(bundle["adt_minus"])
    X_atac = get_norm(bundle["atac"])

    Xtr_raw = [X_rna[tr_idx],  X_adtm[tr_idx],  X_atac[tr_idx]]
    Xva_raw = [X_rna[val_idx], X_adtm[val_idx], X_atac[val_idx]]
    y_tr, y_val = y_all[tr_idx], y_all[val_idx]

    y_mean = float(np.mean(y_tr))
    y_tr_c = y_tr - y_mean
    y_val_c = y_val - y_mean

    Xtr, Xva, preprocs = [], [], []
    for Xm_tr, Xm_va in zip(Xtr_raw, Xva_raw):
        Xm_tr2, Xm_va2, pp = fit_transform_view(Xm_tr, Xm_va, seed=seed)
        Xtr.append(Xm_tr2)
        Xva.append(Xm_va2)
        preprocs.append(pp)

    print(f"\n--- Hyperparameter Selection for {target_name} ---")
    rho_grid = [0.0, 1.0, 5.0, 10.0, 50.0, 100.0]
    lam_grids = [
        [1.0, 10.0, 100.0, 1000.0, 5000.0],     # RNA   (p=2000)
        [0.01, 0.1, 1.0, 10.0, 100.0],           # ADT-minus (p=39)
        [1.0, 10.0, 100.0, 1000.0, 10000.0],     # ATAC  (p=5000)
    ]
    best = holdout_select_params_reg(Xtr, y_tr_c, Xva, y_val_c, rho_grid, lam_grids,
                                     seed=seed, n_jobs=n_jobs)
    print(f"\nGrid Search Complete. Best: {best}")

    print("\n--- Final Model Fit ---")
    Thetas = coop_regression_fit_scaled(
        Xtr, y_tr_c,
        rho=best["rho"],
        lambdas=best["lambdas"],
        X_views_val=Xva,
        y_val=y_val_c,
        max_iter=1000,
        random_state=seed,
    )

    y_pred_c = sum(Xva[m] @ Thetas[m] for m in range(3)).reshape(-1)
    y_pred = y_pred_c + y_mean
    print(f"[DONE] Target: {target_name} | Val MSE: {mean_squared_error(y_val, y_pred):.4f} "
          f"| Val R2: {r2_score(y_val, y_pred):.4f}")

    payload = {
        "Thetas": Thetas,
        "preprocs": preprocs,
        "y_mean": y_mean,
        "params": best,
        "views_order": ["rna", "adt_minus", "atac"],
    }
    joblib.dump(payload, model_root / f"{target_name}_coopreg.pkl")
    print(f"Saved: {model_root / f'{target_name}_coopreg.pkl'}")


if __name__ == "__main__":
    train_regression_pipeline(target_name="CD45RA")
